# Tutorial 1.4: Manual Tracing and Advanced Observability

![](images/5_Advanced-Observability-with-Manual-Tracing.png)
## Custom Instrumentation for Complex Workflows

Welcome to advanced tracing! While autologging is powerful, real-world applications often need custom instrumentation. You'll learn how to trace your own functions and build hierarchical observability into complex GenAI workflows.

### What You'll Learn
- When to use manual tracing vs. autologging
- The `@mlflow.trace` decorator with arguments
- Creating custom spans with proper types
- Adding custom attributes to span
- Tracing agentic workflows with tool usage
- Tracing RAG pipelines end-to-end
- Advanced debugging techniques
- Using Claude Code Assistant for inspecting and debugging

### Prerequisites
- Completed Notebook 1.3 (Introduction to MLflow Tracing)
- Understanding of autologging
- MLflow UI running

### Estimated Time: 30-35 minutes

---
## Step 1: When to Use Manual Tracing

### Autologging is Great For:
- ✅ LLM API calls (OpenAI, Anthropic, Gemmini, etc.)
- ✅ Integrated Framework chains (LangChain, LlamaIndex, LangGraph, etc)
- ✅ Quick prototyping
- ✅ Standard workflows

### Manual Tracing is Needed For:
- ✅ **Custom functions** in your pipeline
- ✅ **Domain-specific operations** (parsing, validation, business logic)
- ✅ **Custom retrievers** or data sources
- ✅ **External API calls** not auto-instrumented
- ✅ **Adding context** not captured automatically
- ✅ **Organizing operations** into logical groups

### Best Practice: Combine Both!
```python
# Use autologging for LLM calls
mlflow.openai.autolog()

# Add manual tracing for custom logic
@mlflow.trace(name="custom_code", span_type=SPAN_TYPE)
def my_custom_function(query):
    # Your custom code
    pass
```

---
## Step 2: Environment Setup

In [1]:
import time

import mlflow
from dotenv import load_dotenv
from utils.clnt_utils import is_databricks_ai_gateway_client, get_databricks_ai_gateway_client, get_openai_client, get_ai_gateway_model_names

# Load environment
load_dotenv()

# Configure MLflow
mlflow.set_tracking_uri("http://localhost:5000")

use_databricks_provider = is_databricks_ai_gateway_client()
if use_databricks_provider:
    client = get_databricks_ai_gateway_client()
    model_name = get_ai_gateway_model_names()[0]
else:
    # Initialize OpenAI
    client = get_openai_client()
    model_name = "gpt-5-mini"
print("✅ Environment configured: using", "Databricks" if use_databricks_provider else "OpenAI", "client")
print(f"   MLflow version: {mlflow.__version__}")
print(f"   Tracking URI: {mlflow.get_tracking_uri()}")

# Create experiment
mlflow.set_experiment("07-manual-tracing")

print("✅ Environment configured")
print("   Ready for manual tracing!")

2026/03/15 16:06:36 INFO mlflow.tracking.fluent: Experiment with name '07-manual-tracing' does not exist. Creating a new experiment.


✅ Environment configured: using OpenAI client
   MLflow version: 3.10.0
   Tracking URI: http://localhost:5000
✅ Environment configured
   Ready for manual tracing!


---
## Step 3: Manual Tracing with Span Types

The `@mlflow.trace` decorator turns any function into a traced span. You control the **name** and **type** of each span, which makes traces searchable and visually organized in the MLflow UI.

### 📋 Standard Span Types

| Type | Use for |
|------|---------|
| `CHAIN` | A sequence of operations (parent wrapper) |
| `LLM` | Language model call |
| `RETRIEVER` | Document retrieval |
| `EMBEDDING` | Text embedding |
| `TOOL` | Tool/function execution |
| `AGENT` | Agent reasoning / planning |
| `PARSER` | Output parsing or business logic |

```python
# Always provide name and span_type
@mlflow.trace(name="my_retriever", span_type="RETRIEVER")
def my_function(x): ...
```

#### Define functions using SPAN types

In [2]:
from typing import List

# Every function uses @mlflow.trace(name="...", span_type="...")

@mlflow.trace(name="db_query_simulation", span_type="CHAIN")
def simulate_custom_query_processing(query: str) -> str:
    """Simulate query preprocessing with an in-memory SQLite DB — CHAIN span."""
    import sqlite3
    conn = sqlite3.connect(":memory:")
    # create a table
    conn.execute("CREATE TABLE q (id INTEGER PRIMARY KEY, raw TEXT, processed TEXT)")
    # process the query
    processed = query.strip().lower()
    # insert the query into the table
    conn.execute("INSERT INTO q (raw, processed) VALUES (?, ?)", (query, processed))
    # commit the transaction
    conn.commit()
    # simulate latency
    time.sleep(0.3)  # Simulate latency
    row = conn.execute("SELECT processed FROM q WHERE id = 1").fetchone()
    conn.close()
    return row[0]

@mlflow.trace(name="preprocess_query", span_type="PARSER")
def preprocess_query(user_query: str) -> str:
    """Parse/clean a query — PARSER span wrapping a CHAIN child span."""
    return simulate_custom_query_processing(user_query)

@mlflow.trace(name="document_retriever", span_type="RETRIEVER")
def retrieve_documents(query: str, top_k: int = 3) -> List[str]:
    """Simulate vector DB retrieval — RETRIEVER span."""

    # simulate latency You would use a real vector DB here
    time.sleep(0.3)
    # simulate retrieval
    docs = [
        "MLflow is an open source platform for the ML and GenAI Applications.",
        "MLflow Tracing provides end-to-end observability for GenAI applications.",
        "MLflow supports experiment tracking, model registry, prompt management, and AI Gateway."
    ]
    return docs[:top_k]

@mlflow.trace(name="get_embedding", span_type="LLM")
def get_embedding(query: str) -> List[float]:
    """Call OpenAI embeddings API — LLM span."""
    response = client.embeddings.create(
        input=query.replace("\n", " "), model="text-embedding-3-small"
    )
    return response.data[0].embedding

@mlflow.trace(name="query_embedder", span_type="EMBEDDING")
def embed_query(query: str) -> List[float]:
    """Embed a query — EMBEDDING parent span with LLM child span."""
    return get_embedding(query)

#### Run the functions and inspect spans

In [3]:
print("\n🔍 Testing manual tracing and span types...\n")

# Plain trace: shows nested spans (preprocess_query → simulate_custom_query_processing)
result = preprocess_query("  What is MLflow?  ")
print(f"preprocess_query output: {result!r}")

query = "What are MLflow's tracing capabilities?"

# Typed span: EMBEDDING parent → LLM child
embedding = embed_query(query)
print(f"\nembed_query output: {embedding[:3]}...")

# Typed span: RETRIEVER
documents = retrieve_documents(query, top_k=2)
print(f"\nretrieve_documents output: {len(documents)} docs")
for i, doc in enumerate(documents, 1):
    print(f"  Doc {i}: {doc[:80]}...")

print("\n✅ All spans created!")
print("""
🔍 In MLflow UI you'll see distinct span types per function:
  preprocess_query          → PARSER
    db_query_simulation     → CHAIN          nested child span
  query_embedder            → EMBEDDING
    get_embedding           → LLM            nested child span
  document_retriever        → RETRIEVER
""")


🔍 Testing manual tracing and span types...

preprocess_query output: 'what is mlflow?'

embed_query output: [0.01396942138671875, 0.0517578125, 0.042724609375]...

retrieve_documents output: 2 docs
  Doc 1: MLflow is an open source platform for the ML and GenAI Applications....
  Doc 2: MLflow Tracing provides end-to-end observability for GenAI applications....

✅ All spans created!

🔍 In MLflow UI you'll see distinct span types per function:
  preprocess_query          → PARSER
    db_query_simulation     → CHAIN          nested child span
  query_embedder            → EMBEDDING
    get_embedding           → LLM            nested child span
  document_retriever        → RETRIEVER



[Trace(trace_id=tr-4ce85407c9f4ac8635cc95a29273b32e), Trace(trace_id=tr-a61f07a4cd19068a21f96d98d7eec337), Trace(trace_id=tr-8b96c365bce27c41a48fd466c38c3513)]

---
## Step 4: Adding Custom Attributes

For additional entities, you can enrich your spans with custom key-value attributes using `mlflow.get_current_active_span()`. 

This lets you attach business-relevant metadata — user IDs, model parameters, retrieval scores — directly to the span.

In [4]:
# Add custom attributes to spans

@mlflow.trace(name="enhanced_retriever", span_type="RETRIEVER")
def enhanced_retriever(query: str, top_k: int = 3) -> List[str]:
    """
    Retriever with rich metadata logging.
    """
    span = mlflow.get_current_active_span()
    # set attributes
    span.set_attributes({"top_k":top_k,
                        "query_length":len(query),
                        "search_method":"cosine_similarity"})

    # Simulate retrieval. Substitute this with a real vector database call.
    time.sleep(0.4)
    docs = [
        
        "MLflow provides tracing capabilities for end-to-end Agentic workflows"
        "MLflow logs all operations and events in a trace, providing a complete view of the workflow."
        "MLflow supports multiple agent frameworks like LangChain, LlamaIndex, and more for tracing"
    ]
    
    # set attributes
    
    span.set_attributes({"num_results": len(docs[:top_k]),
                        "total_docs_in_db": 1000})  # Simulated
    
    return docs[:top_k]

# Test it
print("\n📊 Testing custom attributes...\n")
query = "What is MLflow tracing?"
docs = enhanced_retriever(query, top_k=2)

print("\n✅ Span created with custom attributes!")
print("\n🔍 Custom attributes visible in UI:")
print("   - top_k: 2")
print(f"   - query_length: {len(query)}")
print("   - search_method: vector_similarity")
print("   - num_results: 2")
print("   - total_docs_in_db: 1000")


📊 Testing custom attributes...


✅ Span created with custom attributes!

🔍 Custom attributes visible in UI:
   - top_k: 2
   - query_length: 23
   - search_method: vector_similarity
   - num_results: 2
   - total_docs_in_db: 1000


Trace(trace_id=tr-e99df2757cab974f400f49022dea9fc3)

### 💡 Insight: When to Add Custom Attributes

Add attributes for:
- **Configuration** (top_k, model_name)
- **Performance metrics** (num_results, cache_hit)
- **Data characteristics** (query_length, doc_size)
- **Business logic** (user_tier, feature_flags)
- **Debugging info** (data_source, version)

These make traces **searchable and analyzable**!

---
## Step 5: Tracing Agentic Workflows with Tool Usage

Tool usage is common in agentic systems. Each tool call becomes a child `TOOL` span under an `AGENT` or `CHAIN` parent. 

This lets you see exactly which tools were invoked, with what inputs, and what they returned.

In [5]:
# Agentic workflow with tool usage
from typing import Dict
from typing import List
import json

@mlflow.trace(name="weather_tool", span_type="TOOL")
def get_weather(city: str) -> Dict:
    """Simulated weather tool — replace with a real weather API."""
    span = mlflow.get_current_active_span()
    span.set_attributes({"tool_name": "weather_api", "city": city, "api_version": "v2.0"})

    # simulate latency
    # You would use a real weather API here
    time.sleep(0.2)

    # simulate weather data
    weather_data = {"city": city, "temperature": "72°F", "condition": "Sunny", "humidity": "45%"}
    span.set_attributes({"data_retrieved": True})
    return weather_data


@mlflow.trace(name="create_calendar_invite", span_type="TOOL")
def create_calendar_invite(title: str, date: str, start_time: str, attendees: List[str]) -> Dict:
    """Simulated calendar invite tool — replace with Google Calendar / Outlook API."""
    span = mlflow.get_current_active_span()
    span.set_attributes({
        "tool_name": "create_calendar_invite",
        "title": title,
        "date": date,
        "start_time": start_time,
        "num_attendees": len(attendees)
    })
    time.sleep(0.1)
    return {"status": "created", "invite_id": "inv-20240301-001",
            "title": title, "date": date, "start_time": start_time, "attendees": attendees}


@mlflow.trace(name="send_email", span_type="TOOL")
def send_email(to: str, subject: str, body: str) -> Dict:
    """Simulated email tool — replace with SendGrid / SES / SMTP."""
    span = mlflow.get_current_active_span()
    span.set_attributes({"tool_name": "send_email", "to": to, "subject": subject, "body_length": len(body)})
    time.sleep(0.1)
    return {"status": "sent", "message_id": "msg-20240301-001", "to": to, "subject": subject}


@mlflow.trace(name="generate_random_password", span_type="TOOL")
def generate_random_password(length: int) -> str:
    """Generate a cryptographically strong random password."""
    import secrets
    import string

    span = mlflow.get_current_active_span()
    span.set_attributes({"tool_name": "generate_random_password", "length": length})
    if length < 8:
        raise ValueError("Password length must be at least 8 characters")
    lower, upper, digits, special = (
        string.ascii_lowercase, string.ascii_uppercase,
        string.digits, "!@#$%^&*()_+-=[]{}|;:,.<>?"
    )
    password = [secrets.choice(lower), secrets.choice(upper),
                secrets.choice(digits), secrets.choice(special)]
    all_chars = lower + upper + digits + special
    password += [secrets.choice(all_chars) for _ in range(length - 4)]
    secrets.SystemRandom().shuffle(password)
    return "".join(password)


@mlflow.trace(name="agent_planning", span_type="AGENT")
def agent_plan(user_query: str) -> Dict:
    """LLM decides which tool to use and extracts parameters."""

    span = mlflow.get_current_active_span()
    span.set_attributes({"user_query": user_query})

    # Define the system prompt
    system_prompt = """You are an agent with access to:
1. get_weather(city) - Get weather information
2. create_calendar_invite(title, date, start_time, attendees) - Create a calendar invite; attendees is a list of email strings
3. send_email(to, subject, body) - Send an email
4. generate_random_password(length) - Generate a random password

Decide which tool to use and extract parameters.
Respond with JSON: {"tool": "tool_name", "params": {...}}"""
    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "system", "content": system_prompt},
                  {"role": "user", "content": user_query}],
        response_format={"type": "json_object"},
    )
    plan = json.loads(response.choices[0].message.content)
    span.set_attributes({"selected_tool": plan["tool"]})
    return plan


@mlflow.trace(name="agent_execution", span_type="CHAIN")
def run_agent(user_query: str) -> str:
    """Parent CHAIN: plan → execute tool → generate final response."""
    span = mlflow.get_current_active_span()
    span.set_attributes({"agent_version": "v1.0", "user_query": user_query})

    # Step 1: LLM picks the right tool
    plan = agent_plan(user_query)

    # Step 2: Execute the selected tool
    if plan["tool"] == "get_weather":
        tool_result = get_weather(plan["params"]["city"])
    elif plan["tool"] == "create_calendar_invite":
        tool_result = create_calendar_invite(
            plan["params"]["title"], plan["params"]["date"],
            plan["params"]["start_time"], plan["params"]["attendees"])
    elif plan["tool"] == "send_email":
        tool_result = send_email(
            plan["params"]["to"], plan["params"]["subject"], plan["params"]["body"])
    elif plan["tool"] == "generate_random_password":
        tool_result = generate_random_password(plan["params"]["length"])
    else:
        raise ValueError(f"Unknown tool: {plan['tool']}")

    span.set_attributes({"tool_executed": plan["tool"]})

    # Step 3: LLM summarises the tool result
    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user",
                   "content": f"Tool result: {tool_result}. Answer the user's original request: {user_query}"}],
        max_completion_tokens=1000,
    )
    final_answer = response.choices[0].message.content
    span.set_attributes({"answer_generated": True})
    return final_answer

#### Run the Agentic Workflow

In [6]:
# Test the agent with four realistic queries
print("\n🤖 Testing agentic workflow with tool tracing...\n")

query_1 = "What's the weather in San Francisco?"
answer_1 = run_agent(query_1)
print(f"Query: {query_1}")
print(f"Answer: {answer_1}\n")

query_2 = "Schedule a team meeting called 'Q2 Planning' on 2026-04-01 at 2pm with alice@example.com and bob@example.com"
answer_2 = run_agent(query_2)
print(f"Query: {query_2}")
print(f"Answer: {answer_2}\n")

query_3 = "Send an email to alice@example.com with subject 'Meeting Notes' and body 'Please find the Q2 planning notes attached.'"
answer_3 = run_agent(query_3)
print(f"Query: {query_3}")
print(f"Answer: {answer_3}\n")

query_4 = "Generate a random password with 12 characters"
answer_4 = run_agent(query_4)
print(f"Query: {query_4}")
print("Answer: ************")

print("\n" + "="*60)
print("✅ Agent Workflow Fully Traced!")
print("="*60)

print("""
\n🔍 Agent Trace Hierarchy:

agent_execution (CHAIN)
├── agent_planning (AGENT)
│   └── OpenAI API call (auto-traced)
├── get_weather | create_calendar_invite | send_email | generate_random_password (TOOL)
└── OpenAI API call for final response (auto-traced)

Key insights visible:
✓ Which tool was selected by the LLM
✓ Tool execution time and parameters
✓ Tool result logged as span output
✓ Total agent reasoning time
✓ Token usage across all LLM calls
""")


🤖 Testing agentic workflow with tool tracing...

Query: What's the weather in San Francisco?
Answer: Current conditions in San Francisco: 72°F, Sunny, humidity 45%. 

Would you like an hourly forecast, a multi-day outlook, or any weather alerts?

Query: Schedule a team meeting called 'Q2 Planning' on 2026-04-01 at 2pm with alice@example.com and bob@example.com
Answer: Done — the meeting was scheduled.

- Status: created
- Invite ID: inv-20240301-001
- Title: Q2 Planning
- Date & time: 2026-04-01 at 2:00 PM (14:00)
- Attendees: alice@example.com, bob@example.com

A calendar invite has been created/sent. Would you like to add an agenda, meeting location or conferencing link, set a duration or reminders, or make it recurring?

Query: Send an email to alice@example.com with subject 'Meeting Notes' and body 'Please find the Q2 planning notes attached.'
Answer: Your email has been sent.

- Status: sent
- To: alice@example.com
- Subject: Meeting Notes
- Message ID: msg-20240301-001
- Body: P

[Trace(trace_id=tr-8245c3144110150685b6332f6c970da9), Trace(trace_id=tr-a829e8904ef061d721e9fadd66e3b839), Trace(trace_id=tr-12af1ae339a0ee9eaf7cb97502c26a79), Trace(trace_id=tr-92f3b1eb9e6cbd5ec6a1b401b22b47d2)]

### 🤖 Agent Tracing Benefits

For agentic workflows, tracing reveals:

1. **Decision Making**: Which tool was chosen and why?
2. **Tool Performance**: How long does each tool take?
3. **Error Tracking**: Which tool failed?
4. **Cost Analysis**: How many LLM calls per agent run?
5. **Optimization**: Can we skip unnecessary steps?

**Essential for debugging complex agents!**

---
## Step 6: Complete RAG Pipeline with Hierarchical Tracing

Combine all span types into a single end-to-end RAG pipeline. Each stage — query parsing, embedding, retrieval, prompt formatting, and generation — becomes a typed span under a single parent `CHAIN` span. This gives you full visibility into every step of your pipeline in one trace.

In [ ]:
from typing import Dict, List

@mlflow.trace(name="parse_query", span_type="PARSER")
def parse_query(query: str) -> Dict:
    """Use an LLM to extract intent and entities — traced as a PARSER span."""
    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content":
            f'Return JSON with "intent" (single word) and "entities" (list) for: "{query}"'}],
        response_format={"type": "json_object"},
    )
    result = json.loads(response.choices[0].message.content)
    return {"intent": result.get("intent", "question"),
            "entities": result.get("entities", []),
            "cleaned_query": query.strip()}

@mlflow.trace(name="embed_query", span_type="EMBEDDING")
def embed_query(query: str) -> List[float]:
    """Call the OpenAI embeddings API — traced as an EMBEDDING span."""
    response = client.embeddings.create(
        input=query.replace("\n", " "), model="text-embedding-3-small"
    )
    return response.data[0].embedding

@mlflow.trace(name="vector_search", span_type="RETRIEVER")
def vector_search(embedding: List[float], top_k: int = 3) -> List[str]:
    """Simulate a vector DB lookup — traced as a RETRIEVER span."""
    span = mlflow.get_current_active_span()
    span.set_attributes({"embedding_dim": len(embedding), "top_k": top_k})

    # Simulate retrieval. Substitute this with a real vector database call.
    time.sleep(0.15)
    docs = [
        "MLflow is an open source platform for managing ML and GenAI lifecycle.",
        "MLflow Tracing captures LLM execution with detailed spans.",
        "MLflow integrates with OpenAI, LangChain, and 30+ frameworks.",
    ]
    return docs[:top_k]

@mlflow.trace(name="format_prompt", span_type="PARSER")
def format_prompt(query: str, docs: List[str]) -> str:
    """Build the RAG prompt with retrieved context — traced as a PARSER span."""
    context = "\n".join(f"- {doc}" for doc in docs)
    return f"Use this context to answer:\n\n{context}\n\nQuestion: {query}\n\nAnswer concisely:"

@mlflow.trace(name="generate_answer", span_type="LLM")
def generate_answer(prompt: str) -> str:
    """Call the LLM to produce a final answer — traced as an LLM span."""
    span = mlflow.get_current_active_span()
    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=2000,
    )
    answer = response.choices[0].message.content
    if response.usage:
        span.set_attributes({"tokens_used": response.usage.total_tokens})
    return answer

@mlflow.trace(name="rag_qa_pipeline", span_type="CHAIN")
def rag_qa_pipeline(user_query: str, top_k: int = 3) -> Dict:
    """Parent CHAIN span — wraps the full pipeline end-to-end."""
    span = mlflow.get_current_active_span()
    span.set_attributes({"pipeline_version": "v1.0", "user_query": user_query})

    # start the pipeline as a CHAIN span
    #  Step 1: Parse the query   
    parsed    = parse_query(user_query)

    # Step 2: Embed the query
    embedding = embed_query(parsed["cleaned_query"])

    # Step 3: Search the vector database
    docs      = vector_search(embedding, top_k=top_k)

    # Step 4: Format the prompt
    prompt    = format_prompt(user_query, docs)

    # Step 5: Generate the answer
    answer    = generate_answer(prompt)

    span.set_attributes({"num_docs": len(docs), "answer_generated": True})
    return {"query": user_query, "answer": answer, "num_docs": len(docs)}

#### Run the end-to-end RAG Pipeline

In [ ]:
# Test the complete RAG pipeline
print("\n🤖 Testing complete RAG pipeline...\n")

result = rag_qa_pipeline("What tracing capabilities does MLflow provide?", top_k=3)

print(f"Query:  {result['query']}")
print(f"\nDocs used: {result['num_docs']}")
print(f"\nAnswer:\n{result['answer']}")

print("\n" + "="*60)
print("✅ Full RAG pipeline traced!")
print("="*60)
print("""
🔍 Trace hierarchy in MLflow UI:

rag_qa_pipeline  (CHAIN)
├── parse_query  (PARSER)
├── embed_query  (EMBEDDING)
├── vector_search  (RETRIEVER)
├── format_prompt  (PARSER)
└── generate_answer  (LLM)

Each span shows inputs, outputs, duration, and custom attributes.
""")

### 💡 Why Hierarchical Tracing Matters

Within this single trace you can immediately examine all the spans and answer:
- **Which step is slowest?** (compare span durations in the timeline)
- **What did the query parser extract?** (inspect `parse_query` inputs/outputs)
- **Which documents were retrieved?** (check `vector_search` output)
- **How many tokens did the answer cost?** (`tokens_used` attribute on `generate_answer`)
- **Where did it fail?** (failed span is highlighted — all attributes logged before the error are preserved)

---
## Step 7: Advanced Debugging Techniques

Traces make bugs easy to find. Let's see how to use span data to identify issues in your GenAI application.

In [ ]:
# Debugging example: Trace a buggy function, which can raise an error

@mlflow.trace(name="buggy_processor", span_type="PARSER")
def process_with_validation(data: Dict) -> Dict:
    """
    Fake function with potential errors.
    """
    span = mlflow.get_current_active_span()
    span.set_attributes({
        "input_keys": list(data.keys())
    })
    
    # Log intermediate state for debugging
    span.set_attributes({
        "validation_step": "checking_required_fields"
    })
    
    # This might raise an error
    if "required_field" not in data:
        span.set_attributes({
            "error_type": "missing_field"
        })
        raise ValueError("Missing required_field")
    span.set_attributes({
        "validation_step": "passed"
    })
    
    # Process data
    result = {"processed": True, "value": data.get("required_field")}
    
    return result

print("\n🐛 Testing error tracing...\n")

# Success case
try:
    result = process_with_validation({"required_field": "test"})
    print(f"✅ Success case: {result}")
except Exception as e:
    print(f"❌ Error: {e}")

# Failure case
print("\nTesting failure case...")
try:
    result = process_with_validation({"wrong_field": "test"})
    print(f"✅ Success case: {result}")
except Exception as e:
    print(f"❌ Error: {e}")
    print("\n🔍 Error details captured in trace!")

print("""
\n💡 Debugging with Traces:

1. Filter traces by error status in UI
2. View the failed span (marked in red)
3. Check custom attributes:
   - What were the inputs?
   - What validation step failed?
   - What was the error type?
4. Compare with successful traces
5. Reproduce the issue with exact inputs
6. Use MLflow Assistant to get insights on the failed traces and root cause

Attributes logged BEFORE the error help identify root cause!
""")

---
## Step 8: Performance Analysis

Use traces to identify bottlenecks. The MLflow UI shows span durations visually — making it easy to see which steps take the most time.

In [ ]:
# Performance analysis example
print("""
╔══════════════════════════════════════════════════════════════╗
║         Performance Analysis with Traces                     ║
╚══════════════════════════════════════════════════════════════╝

In the MLflow UI, you can:

1. 📊 TIMELINE VIEW:
   - See which operations take the most time
   - Identify serial vs parallel operations
   - Find bottlenecks visually

2. 📈 AGGREGATE METRICS:
   - Average latency per span type
   - P50, P95, P99 latencies
   - Success rate per operation

3. 🔍 COMPARISON:
   - Compare traces before/after optimization
   - A/B test different implementations
   - Track performance over time

4. 💡 OPTIMIZATION STRATEGIES:
   
   If retrieval is slow:
   - Check embedding generation time
   - Optimize vector search
   - Optimize prompt with GEPA
   
   If LLM calls are slow:
   - Reduce max_tokens
   - Use streaming responses
   - Try smaller models to reduce latency or curb costs
   
   If overall latency is high:
   - Parallelize independent operations
   - Cache frequent queries
   - Optimize prompt length

5. 📊 METRICS TO TRACK:
   - End-to-end latency
   - Per-operation latency
   - Token usage and cost
   - Error rate

""")

---
## Summary

In this notebook, you learned:

1. ✅ When to use manual tracing vs. autologging
2. ✅ The `@mlflow.trace` decorator for custom functions
3. ✅ Span types (`CHAIN`, `LLM`, `RETRIEVER`, `EMBEDDING`, `TOOL`, `AGENT`, `PARSER`) and naming conventions
4. ✅ Adding custom attributes with `span.set_attributes({...})`
5. ✅ Building hierarchical traces — a complete RAG pipeline end-to-end
6. ✅ Tracing agentic workflows with LLM-based tool selection
7. ✅ Advanced debugging with error traces
8. ✅ Performance analysis and optimization strategies

### Key Takeaways

- **Combine** autologging and manual tracing for complete coverage
- **Use span types** for better organization and filtering in the UI
- **Add custom attributes** for debugging and analysis
- **Build CHAIN hierarchies** to represent complex workflows accurately
- **Trace everything** from retrieval to generation to validation

### Production Best Practices

1. Always trace in development and staging
2. Monitor key metrics from traces (latency, errors, costs)
3. Set up alerts for anomalies
4. Review traces during incident response

### What's Next?

**📓 Notebook 1.5: Prompt Management and Optimization**

Learn how to:
- Create reusable prompt templates
- Version prompts systematically
- Link prompts to experiments and traces
- Share prompts across your team
- Optimize prompts with GEPA